[<img src="https://colab.research.google.com/assets/colab-badge.svg" align="left" alt="Open In Colab"/>](https://colab.research.google.com/github/luiscuellarh/apigee-unified-ai-gateway/blob/main/notebooks/03_apigee_mcp_gateway.ipynb)

# Apigee AI Gateway — Model Context Protocol (MCP) Gateway Lab

## Overall Description
Welcome to the **Apigee MCP Gateway Interactive Lab**.

As organizations connect AI agents to internal business systems—such as **Loans** and **Payments** services—using the **Model Context Protocol (MCP)**, they need a secure, central control point in front of those tools. Without an API gateway, AI agents might call backend services without proper authentication, exceed rate limits, or invoke unauthorized tools.

### What This Notebook Does
This notebook walks you through deploying and testing an enterprise **MCP Gateway on Google Cloud Apigee** using the **Apigee Feature Templater (AFT)**:
1. **Validates Your Cloud Environment (Step 0):** Checks your Google Cloud project, Apigee organization, IAM permissions, Gemini model access, and installs the deployment toolkit in a single click.
2. **Secures a Loans MCP Service with OAuth 2.0 (Step 1):** Deploys an Apigee MCP proxy for a Loans service, packages it into an API Product, registers a Developer Application, and launches an interactive **Loans MCP Web Client** so you can discover and test loan tools live.
3. **Secures a Payments MCP Service with API Keys & OAuth 2.0 (Step 2):** Deploys an Apigee MCP proxy for a Payments service, configures access credentials, and launches an interactive **Payments MCP Web Client** to inspect and invoke payment tools in real time.

### Business Value & Expected Outcome
- **Enterprise Security:** AI agents authenticate with OAuth 2.0 tokens or API keys at the Apigee edge while Apigee securely authenticates to backend Cloud Run MCP servers using Google ID tokens.
- **Traffic & Tool Governance:** Apigee inspects MCP JSON-RPC 2.0 payloads (`initialize`, `tools/list`, `tools/call`) and enforces quotas and rate limits before requests ever reach your backend.
- **Expected Outcome:** By the end of this notebook, you will have two production-ready MCP Gateways (`loanservice` and `paymentservice`) running in your Apigee environment and two interactive web clients inside Colab to explore and execute MCP tools end-to-end.

---

## Step 0 — Environment Setup & Preflight Verification

### What This Step Does
Checks that your Google Cloud project, Apigee instance, Service Account permissions, live Gemini model connection, and the Apigee Feature Templater (`aft`) CLI are properly configured, and downloads the gateway templates repository.

### Why It Is Necessary
Before deploying any MCP proxies or developer applications, we must confirm that your Apigee runtime is reachable, your account has permission to deploy proxies, and all required configuration files are available locally. Catching missing settings upfront prevents partial deployments later.

### Inputs & Outputs
- **What You Need to Do (Inputs):** Fill in the form fields below—specifically `PROJECT_ID`, `APIGEE_ORG` (leave blank to default to `PROJECT_ID`), `HOSTNAME` (your Apigee nip.io or custom domain), and optionally `SERVICE_ACCOUNT`—then click **Run**.
- **What You Will Get (Outputs):** A green **`PREFLIGHT PASSED — safe to proceed`** summary table confirming your project number, Apigee environment, active Service Account, Gemini connectivity, and cloned template repository.

In [ ]:
#@title 0.1–0.4 Step 0 — Environment Setup & Preflight Verification
# Runs 0.1 (Inputs), 0.2 (Apigee Topology & IAM), 0.3 (Service Account & Gemini),
# and 0.4 (AFT CLI, Template Repo & Summary Gate) sequentially in a single execution.

import os
import pathlib
import re
import shutil
import subprocess
import time
from typing import Any, Dict, List, Optional, Tuple
import requests

# ============================================================================
# 0.1 Inputs & Global Configuration
# ============================================================================
print("▶ [0.1] Validating Inputs & Global Configuration...")
try:
    from google.colab import auth
    auth.authenticate_user()
    print("✅ Authenticated with Google Cloud in Colab.")
except ImportError:
    print("ℹ️ Running in a local or pre-authenticated Jupyter environment.")

#@markdown ### Project & Apigee
PROJECT_ID = "" #@param {type:"string"}
APIGEE_ORG = "" #@param {type:"string"}
APIGEE_ENV = "dev" #@param {type:"string"}
ENV_GROUP = "dev-group" #@param {type:"string"}
REGION = "europe-west1" #@param {type:"string"}
HOSTNAME = "" #@param {type:"string"}

#@markdown ### Service Account
SERVICE_ACCOUNT = "" #@param {type:"string"}

#@markdown ### Gemini
GEMINI_MODEL = "gemini-2.5-flash" #@param {type:"string"}

#@markdown ### Behaviour Flags
ENABLE_APIS = False #@param {type:"boolean"}

PROJECT_ID = PROJECT_ID.strip()
APIGEE_ENV = APIGEE_ENV.strip()
APIGEE_ENVIRONMENT = APIGEE_ENV
ENV_GROUP = ENV_GROUP.strip()
REGION = REGION.strip()
HOSTNAME = HOSTNAME.strip()
SERVICE_ACCOUNT = SERVICE_ACCOUNT.strip()
APIGEE_ORG = APIGEE_ORG.strip() or PROJECT_ID
GEMINI_MODEL = GEMINI_MODEL.strip().removeprefix("google/")

required_inputs = {
    "PROJECT_ID": PROJECT_ID,
    "APIGEE_ENV": APIGEE_ENV,
    "REGION": REGION,
    "GEMINI_MODEL": GEMINI_MODEL,
}
missing_inputs = [name for name, val in required_inputs.items() if not val]
if missing_inputs:
    raise SystemExit(f"❌ HALT: Missing required parameter(s): {', '.join(missing_inputs)}.")

sa_provided = bool(SERVICE_ACCOUNT)
if SERVICE_ACCOUNT and "@" not in SERVICE_ACCOUNT:
    SERVICE_ACCOUNT = f"{SERVICE_ACCOUNT}@{PROJECT_ID}.iam.gserviceaccount.com"
if SERVICE_ACCOUNT and not re.fullmatch(
    r"[a-zA-Z0-9._-]+@[a-zA-Z0-9.-]+\.gserviceaccount\.com", SERVICE_ACCOUNT
):
    raise SystemExit(
        f"❌ HALT: Invalid SERVICE_ACCOUNT '{SERVICE_ACCOUNT}' "
        "(expected *.gserviceaccount.com)."
    )

subprocess.run(
    ["gcloud", "config", "set", "project", PROJECT_ID, "--quiet"],
    check=True,
    capture_output=True,
)

APIGEE_BASE = f"https://apigee.googleapis.com/v1/organizations/{APIGEE_ORG}"
REPO_URL = "https://github.com/luiscuellarh/apigee-unified-ai-gateway.git"
REPO_BRANCH = "main"
REPO_DIR = pathlib.Path.cwd() / "apigee-unified-ai-gateway"

sa_summary = (
    f"{SERVICE_ACCOUNT} (provided — verify only)"
    if sa_provided
    else f"apigee-ai-gateway-sa@{PROJECT_ID}.iam.gserviceaccount.com (auto-provision)"
)
print(
    f"✅ [0.1] Inputs ready | project={PROJECT_ID} | org={APIGEE_ORG} | "
    f"env={APIGEE_ENV} ({ENV_GROUP}) | region={REGION} | sa={sa_summary}"
)


# ============================================================================
# Shared Cloud & AFT Deployment Helpers
# ============================================================================
def get_gcloud_token() -> str:
    """Return the active Google Cloud OAuth access token."""
    return subprocess.check_output(
        ["gcloud", "auth", "print-access-token"], text=True
    ).strip()


def gcp_request(
    method: str,
    url: str,
    json_body: Optional[Dict[str, Any]] = None,
    headers: Optional[Dict[str, str]] = None,
) -> Tuple[int, Dict[str, Any]]:
    """Execute an authenticated UTF-8 JSON request against a Google Cloud API."""
    req_headers = {
        "Authorization": f"Bearer {get_gcloud_token()}",
        "Content-Type": "application/json; charset=utf-8",
        "Accept": "application/json; charset=utf-8",
    }
    if headers:
        req_headers.update(headers)
    try:
        resp = requests.request(
            method, url, json=json_body, headers=req_headers, timeout=30
        )
        resp.encoding = "utf-8"
        return resp.status_code, (resp.json() if resp.text else {})
    except ValueError:
        return resp.status_code, {"raw": resp.text}
    except Exception as exc:
        return 0, {"error": str(exc), "url": url}


def wait_for_proxy_ready(
    proxy_name: str,
    env_name: Optional[str] = None,
    timeout_sec: int = 180,
) -> str:
    """Poll Apigee until the latest revision of proxy_name reaches READY state."""
    target_env = env_name or APIGEE_ENV
    list_url = f"{APIGEE_BASE}/environments/{target_env}/apis/{proxy_name}/deployments"
    deadline = time.time() + timeout_sec
    last_rev, last_state = None, "NOT_DEPLOYED"

    print(f"⏳ Waiting for proxy '{proxy_name}' to reach READY state in '{target_env}'...")
    while time.time() < deadline:
        status_code, data = gcp_request("GET", list_url)
        deployments = data.get("deployments", []) if status_code == 200 else []
        if deployments:
            latest = max(deployments, key=lambda d: int(d.get("revision", 0)))
            last_rev = latest.get("revision")
            rev_url = (
                f"{APIGEE_BASE}/environments/{target_env}/apis/{proxy_name}"
                f"/revisions/{last_rev}/deployments"
            )
            rev_status, rev_data = gcp_request("GET", rev_url)
            last_state = (
                rev_data.get("state", latest.get("state", "UNKNOWN"))
                if rev_status == 200
                else "UNKNOWN"
            )
            if last_state == "READY":
                print(
                    f"✅ Proxy '{proxy_name}' (revision {last_rev}) is READY "
                    f"in env '{target_env}'."
                )
                return str(last_rev)
            print(f"   • proxy={proxy_name} | rev={last_rev} | state={last_state} — waiting 5s...")
        else:
            print(f"   • proxy={proxy_name} | state=WAITING_FOR_DEPLOYMENT — waiting 5s...")
        time.sleep(5)

    raise SystemExit(
        f"❌ HALT: Timed out waiting for proxy '{proxy_name}' in env '{target_env}' "
        f"(last revision={last_rev}, state={last_state})."
    )


def ensure_repo_files(relative_paths: List[str]) -> None:
    """Ensure all required YAML files exist in REPO_DIR, fetching from Git if needed."""
    if any(not (REPO_DIR / rel_p).is_file() for rel_p in relative_paths):
        print(f"🔄 Fetching latest files from origin/{REPO_BRANCH}...")
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH],
            capture_output=True,
            text=True,
        )
        for rel_p in relative_paths:
            if not (REPO_DIR / rel_p).is_file():
                subprocess.run(
                    ["git", "-C", str(REPO_DIR), "checkout", f"origin/{REPO_BRANCH}", "--", rel_p],
                    capture_output=True,
                    text=True,
                )
    for rel_p in relative_paths:
        if not (REPO_DIR / rel_p).is_file():
            raise SystemExit(
                f"❌ HALT: Required file not found at '{REPO_DIR / rel_p}'. "
                "Please re-run Step 0."
            )


def ensure_env_in_product_yaml(product_rel_path: str, env_name: Optional[str] = None) -> None:
    """Ensure the target Apigee environment is listed under 'environments:' in a product YAML."""
    target_env = env_name or APIGEE_ENV
    product_path = REPO_DIR / product_rel_path
    content = product_path.read_text(encoding="utf-8")
    if not re.search(rf"(?m)^\s*-\s*{re.escape(target_env)}\s*$", content):
        updated = re.sub(
            r"(?m)^(\s*environments:\s*\n)",
            rf"\g<1>  - {target_env}\n",
            content,
            count=1,
        )
        product_path.write_text(updated, encoding="utf-8")


def run_aft_deploy(rel_path: str, step_label: str) -> None:
    """Execute an Apigee Feature Templater (AFT) deployment command."""
    os.environ.pop("AFT_REPOSITORY", None)
    aft_executable = shutil.which("aft") or "aft"
    cmd = [
        aft_executable,
        "-i", rel_path,
        "--organization", APIGEE_ORG,
        "--environment", APIGEE_ENV,
        "--sa", SERVICE_ACCOUNT,
    ]
    print(f"\n▶ {step_label}: {' '.join(cmd)}")
    result = subprocess.run(cmd, cwd=str(REPO_DIR), capture_output=True, text=True)
    if result.returncode != 0:
        error_detail = (result.stderr or result.stdout).strip()
        raise SystemExit(
            f"❌ HALT: AFT deployment failed for '{rel_path}' "
            f"(exit {result.returncode}):\n{error_detail}"
        )
    if result.stdout.strip():
        print(result.stdout.strip())


# ============================================================================
# 0.2 Verify Apigee Topology, APIs & IAM
# ============================================================================
print("\n▶ [0.2] Verifying Apigee Topology, APIs & IAM...")
active_identity = subprocess.check_output(
    ["gcloud", "config", "get-value", "account"], text=True
).strip()
if not active_identity:
    _, token_info = gcp_request(
        "GET", f"https://oauth2.googleapis.com/tokeninfo?access_token={get_gcloud_token()}"
    )
    active_identity = token_info.get("email", "")
if not active_identity:
    raise SystemExit("❌ HALT: Could not resolve active Google Cloud caller identity.")

caller_prefix = (
    "serviceAccount" if active_identity.endswith(".gserviceaccount.com") else "user"
)
print(f"✅ Active identity: {active_identity}")

REQUIRED_SERVICES = [
    "apigee.googleapis.com",
    "cloudresourcemanager.googleapis.com",
    "iamcredentials.googleapis.com",
    "aiplatform.googleapis.com",
]
disabled_services = [
    svc
    for svc in REQUIRED_SERVICES
    if gcp_request(
        "GET", f"https://serviceusage.googleapis.com/v1/projects/{PROJECT_ID}/services/{svc}"
    )[1].get("state")
    != "ENABLED"
]
if disabled_services:
    if ENABLE_APIS or disabled_services == ["iamcredentials.googleapis.com"]:
        gcp_request(
            "POST",
            f"https://serviceusage.googleapis.com/v1/projects/{PROJECT_ID}/services:batchEnable",
            {"serviceIds": disabled_services},
        )
        time.sleep(3)
    else:
        raise SystemExit(
            f"❌ HALT: Disabled API(s): {', '.join(disabled_services)}. "
            "Set ENABLE_APIS=True in Step 0 and re-run."
        )
print(f"✅ Required APIs ENABLED: {', '.join(REQUIRED_SERVICES)}")

org_status, org_data = gcp_request("GET", APIGEE_BASE)
if org_status != 200 or org_data.get("state") != "ACTIVE":
    raise SystemExit(
        f"❌ HALT: Apigee org '{APIGEE_ORG}' is not ACTIVE (HTTP {org_status}): {org_data}"
    )

env_status, env_data = gcp_request("GET", f"{APIGEE_BASE}/environments/{APIGEE_ENV}")
if env_status != 200:
    raise SystemExit(
        f"❌ HALT: Apigee env '{APIGEE_ENV}' not found in '{APIGEE_ORG}' "
        f"(HTTP {env_status}): {env_data}"
    )

_, envgroups_resp = gcp_request("GET", f"{APIGEE_BASE}/envgroups")
matched_group, matched_hosts = None, []
for group in envgroups_resp.get("environmentGroups", []):
    group_name = group["name"]
    if ENV_GROUP and group_name != ENV_GROUP:
        continue
    _, attach_resp = gcp_request("GET", f"{APIGEE_BASE}/envgroups/{group_name}/attachments")
    attached_envs = [
        att.get("environment") for att in attach_resp.get("environmentGroupAttachments", [])
    ]
    if APIGEE_ENV in attached_envs:
        matched_group = group_name
        matched_hosts = group.get("hostnames", [])
        break

if not matched_group or (not HOSTNAME and not matched_hosts):
    raise SystemExit(
        f"❌ HALT: Environment '{APIGEE_ENV}' is not attached to an environment group "
        f"with a valid hostname in '{APIGEE_ORG}'."
    )
ENV_GROUP = matched_group
HOSTNAME = HOSTNAME or matched_hosts[0]
print(
    f"✅ Apigee Topology: org={APIGEE_ORG} | env={APIGEE_ENV} | "
    f"envgroup={ENV_GROUP} | host={HOSTNAME}"
)

REQUIRED_APIGEE_PERMS = [
    "apigee.proxies.create", "apigee.proxies.get", "apigee.proxies.update", "apigee.proxies.delete",
    "apigee.proxyrevisions.get", "apigee.proxyrevisions.update", "apigee.proxyrevisions.delete",
    "apigee.proxyrevisions.deploy", "apigee.proxyrevisions.undeploy",
    "apigee.deployments.create", "apigee.deployments.get", "apigee.deployments.update", "apigee.deployments.delete",
    "apigee.sharedflows.create", "apigee.sharedflows.get", "apigee.sharedflows.delete",
    "apigee.sharedflowrevisions.update", "apigee.sharedflowrevisions.deploy", "apigee.sharedflowrevisions.undeploy",
    "apigee.keyvaluemaps.create", "apigee.keyvaluemaps.get", "apigee.keyvaluemaps.update", "apigee.keyvaluemaps.delete",
    "apigee.keyvaluemapentries.create", "apigee.keyvaluemapentries.get", "apigee.keyvaluemapentries.update", "apigee.keyvaluemapentries.delete",
    "apigee.targetservers.create", "apigee.targetservers.get", "apigee.targetservers.update", "apigee.targetservers.delete",
]
_, iam_resp = gcp_request(
    "POST",
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}:testIamPermissions",
    {"permissions": REQUIRED_APIGEE_PERMS},
)
granted_perms = set(iam_resp.get("permissions", []))
missing_perms = [perm for perm in REQUIRED_APIGEE_PERMS if perm not in granted_perms]
if missing_perms:
    raise SystemExit(f"❌ HALT: Missing Apigee IAM permissions: {', '.join(missing_perms)}")
print(f"✅ [0.2] Verified all {len(REQUIRED_APIGEE_PERMS)} Apigee IAM permissions.")


# ============================================================================
# 0.3 Verify Service Account Roles, Bindings & Live Gemini
# ============================================================================
print("\n▶ [0.3] Verifying Service Account Roles, Bindings & Live Gemini...")


def ensure_iam_bindings(
    get_url: str,
    set_url: str,
    required_pairs: List[Tuple[str, str]],
    label: str,
) -> None:
    """Ensure all (member, role) pairs exist on the target IAM policy."""
    status_code, policy = gcp_request("POST", get_url, {})
    if status_code != 200:
        raise SystemExit(
            f"❌ HALT: Failed to read {label} IAM policy (HTTP {status_code}): {policy}"
        )
    bindings = policy.setdefault("bindings", [])
    policy_changed = False
    for member, role in required_pairs:
        entry = next(
            (b for b in bindings if b.get("role") == role and not b.get("condition")),
            None,
        )
        if entry and member in entry.get("members", []):
            print(f"  ✔ {label} verified: {role} -> {member}")
        else:
            print(f"  ⏳ Binding {label}: {role} -> {member}")
            if entry:
                entry.setdefault("members", []).append(member)
            else:
                bindings.append({"role": role, "members": [member]})
            policy_changed = True

    if policy_changed:
        set_status, set_resp = gcp_request("POST", set_url, {"policy": policy})
        if set_status != 200:
            raise SystemExit(
                f"❌ HALT: Failed to update {label} IAM policy (HTTP {set_status}): {set_resp}"
            )
        print(f"  ✔ {label} bindings updated.")


_, proj_data = gcp_request(
    "GET", f"https://cloudresourcemanager.googleapis.com/v1/projects/{APIGEE_ORG}"
)
if not proj_data.get("projectNumber"):
    _, proj_data = gcp_request(
        "GET", f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}"
    )
if not proj_data.get("projectNumber"):
    raise SystemExit(f"❌ HALT: Unable to resolve projectNumber for '{PROJECT_ID}': {proj_data}")

PROJECT_NUMBER = str(proj_data["projectNumber"])
apigee_service_agent = f"service-{PROJECT_NUMBER}@gcp-sa-apigee.iam.gserviceaccount.com"

target_sa = SERVICE_ACCOUNT or f"apigee-ai-gateway-sa@{PROJECT_ID}.iam.gserviceaccount.com"
sa_status, sa_data = gcp_request(
    "GET", f"https://iam.googleapis.com/v1/projects/-/serviceAccounts/{target_sa}"
)

if sa_provided:
    if sa_status != 200 or sa_data.get("disabled", False):
        raise SystemExit(
            f"❌ HALT: Provided SERVICE_ACCOUNT '{SERVICE_ACCOUNT}' is missing or disabled "
            f"(HTTP {sa_status})."
        )
    sa_lifecycle_state = "VERIFIED_EXISTING"
elif sa_status == 200:
    SERVICE_ACCOUNT = target_sa
    sa_lifecycle_state = "REUSED_DEFAULT"
elif sa_status == 404:
    create_status, create_data = gcp_request(
        "POST",
        f"https://iam.googleapis.com/v1/projects/{PROJECT_ID}/serviceAccounts",
        {
            "accountId": "apigee-ai-gateway-sa",
            "serviceAccount": {"displayName": "Apigee AI Gateway Service Account"},
        },
    )
    if create_status not in (200, 201):
        raise SystemExit(
            f"❌ HALT: Failed to create default service account '{target_sa}' "
            f"(HTTP {create_status}): {create_data}"
        )
    SERVICE_ACCOUNT = create_data.get("email", target_sa)
    sa_lifecycle_state = "CREATED_DEFAULT"
    time.sleep(3)
else:
    raise SystemExit(
        f"❌ HALT: Unable to inspect service account '{target_sa}' (HTTP {sa_status}): {sa_data}"
    )

proj_iam_url = f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}"
ensure_iam_bindings(
    f"{proj_iam_url}:getIamPolicy",
    f"{proj_iam_url}:setIamPolicy",
    [
        (f"serviceAccount:{SERVICE_ACCOUNT}", "roles/aiplatform.user"),
        (f"serviceAccount:{SERVICE_ACCOUNT}", "roles/iam.serviceAccountTokenCreator"),
    ],
    "Project role",
)

sa_iam_url = f"https://iam.googleapis.com/v1/projects/-/serviceAccounts/{SERVICE_ACCOUNT}"
ensure_iam_bindings(
    f"{sa_iam_url}:getIamPolicy",
    f"{sa_iam_url}:setIamPolicy",
    [
        (f"serviceAccount:{apigee_service_agent}", "roles/iam.serviceAccountTokenCreator"),
        (f"serviceAccount:{SERVICE_ACCOUNT}", "roles/iam.serviceAccountTokenCreator"),
        (f"serviceAccount:{SERVICE_ACCOUNT}", "roles/iam.serviceAccountUser"),
        (f"{caller_prefix}:{active_identity}", "roles/iam.serviceAccountUser"),
    ],
    "SA <GoogleAccessToken> binding",
)
print(f"✅ Service Account ready: {SERVICE_ACCOUNT} ({sa_lifecycle_state})")

gemini_payload = {
    "contents": [{"role": "user", "parts": [{"text": "Reply with READY."}]}],
    "generationConfig": {"maxOutputTokens": 8, "temperature": 0.0},
}


def invoke_gemini(location: str) -> Tuple[int, Dict[str, Any]]:
    """Test live Vertex AI Gemini model connectivity in the specified location."""
    host = (
        "aiplatform.googleapis.com"
        if location == "global"
        else f"{location}-aiplatform.googleapis.com"
    )
    url = (
        f"https://{host}/v1/projects/{PROJECT_ID}/locations/{location}"
        f"/publishers/google/models/{GEMINI_MODEL}:generateContent"
    )
    status_code, resp_data = gcp_request(
        "POST", url, gemini_payload, headers={"x-goog-user-project": PROJECT_ID}
    )
    return (
        gcp_request("POST", url, gemini_payload)
        if status_code == 403
        else (status_code, resp_data)
    )


vertex_location, gemini_status, gemini_data = REGION, 0, {}
for _ in range(12):
    gemini_status, gemini_data = invoke_gemini(REGION)
    if gemini_status == 200:
        vertex_location = REGION
        break
    for fallback_loc in [loc for loc in ("global", "us-central1") if loc != REGION]:
        fb_status, fb_data = invoke_gemini(fallback_loc)
        if fb_status == 200:
            gemini_status, gemini_data, vertex_location = fb_status, fb_data, fallback_loc
            break
    if gemini_status == 200 or gemini_status not in (403, 404):
        break
    time.sleep(5)

if gemini_status != 200:
    raise SystemExit(f"❌ HALT: Live Gemini call failed (HTTP {gemini_status}): {gemini_data}")
print(f"✅ [0.3] Live Gemini verified: model={GEMINI_MODEL} | location={vertex_location}")


# ============================================================================
# 0.4 Install AFT CLI, Clone Repository & Summary Gate
# ============================================================================
print("\n▶ [0.4] Installing AFT CLI & Synchronizing Template Repository...")
os.environ.pop("AFT_REPOSITORY", None)
for bin_dir in [
    str(pathlib.Path.home() / ".aft" / "bin"),
    str(pathlib.Path.home() / ".local" / "bin"),
]:
    if bin_dir not in os.environ["PATH"]:
        os.environ["PATH"] = f"{bin_dir}:{os.environ['PATH']}"

if not shutil.which("aft"):
    subprocess.run(
        "curl -fsSL https://raw.githubusercontent.com/apigee/apigee-templater/main/install.sh | sh",
        shell=True,
        check=True,
    )

aft_bin = shutil.which("aft")
if not aft_bin:
    raise SystemExit("❌ HALT: 'aft' binary not found on PATH after installation.")
aft_version = subprocess.run(
    [aft_bin, "--version"], capture_output=True, text=True, check=True
).stdout.strip()
print(f"✅ AFT CLI ready: {aft_version} ({aft_bin})")

if (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_BRANCH],
        check=True,
        capture_output=True,
        text=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"],
        check=True,
        capture_output=True,
        text=True,
    )
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
        capture_output=True,
        text=True,
    )

if not (REPO_DIR / "templates").is_dir() or not (REPO_DIR / "features").is_dir():
    raise SystemExit(
        f"❌ HALT: Cloned repository at '{REPO_DIR}' is missing 'templates/' or 'features/'."
    )

os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "PROJECT_ID": PROJECT_ID,
    "APIGEE_ORG": APIGEE_ORG,
    "APIGEE_ENV": APIGEE_ENV,
    "APIGEE_ENVIRONMENT": APIGEE_ENVIRONMENT,
    "ENV_GROUP": ENV_GROUP,
    "REGION": REGION,
    "HOSTNAME": HOSTNAME,
    "SERVICE_ACCOUNT": SERVICE_ACCOUNT,
    "GEMINI_MODEL": GEMINI_MODEL,
    "VERTEX_LOCATION": vertex_location,
    "REPO_DIR": str(REPO_DIR),
})

num_templates = len(list((REPO_DIR / "templates").glob("*.yaml")))
num_features = len(list((REPO_DIR / "features").glob("*.yaml")))

print("=" * 72)
for label, value in [
    ("GCP Project / No.", f"{PROJECT_ID} ({PROJECT_NUMBER})"),
    ("Apigee Org / Env", f"{APIGEE_ORG} / {APIGEE_ENV} ({ENV_GROUP})"),
    ("Gateway Hostname", HOSTNAME),
    ("Active Identity", active_identity),
    ("Service Account", f"{SERVICE_ACCOUNT} ({sa_lifecycle_state})"),
    ("Gemini Readiness", f"PASS ({GEMINI_MODEL} @ {vertex_location})"),
    ("AFT Toolkit", f"READY ({aft_version})"),
    ("Local Repo (AFT)", f"{REPO_DIR.name} ({num_templates} templates, {num_features} features)"),
]:
    print(f"{label:<20}: {value}")
print("=" * 72)
print("PREFLIGHT PASSED — safe to proceed")

## Step 1.1 — Deploy the Loans MCP Gateway, API Product & Developer App

### What This Step Does
Deploys the **Loans MCP Gateway Proxy** (`mcp-gateway-loan-product`), the **OAuth 2.0 Token Server** (`auth-oauth-server`), the **Loans API Product** (`ai-loan-agent-package`), and the **Loans Developer Application** (`Loan Agent App`) into your Apigee environment using the Apigee Feature Templater (`aft`).

### Why It Is Necessary
1. **The MCP Gateway Proxy** acts as a protective front door at `/loanservice/mcp`. It validates OAuth 2.0 tokens, parses incoming MCP JSON-RPC 2.0 messages, and securely forwards approved requests to the backend Cloud Run Loans MCP server.
2. **The API Product** defines the access rules, allowed MCP operations, and traffic quotas for loan agents.
3. **The Developer App** issues the OAuth 2.0 `Client ID` and `Client Secret` that an AI agent (or our interactive test client in Step 1.2) uses to authenticate.

### Inputs & Outputs
- **What You Need to Do (Inputs):** Simply click **Run** on the code cell below. It automatically uses the project, environment, and Service Account validated in Step 0.
- **What You Will Get (Outputs):** Confirmation that the OAuth 2.0 Server and `mcp-gateway-loan-product` proxy are deployed and in the `READY` state, along with the live URLs for the OAuth token endpoint (`/v1/oauth/token`) and Loans MCP endpoint (`/loanservice/mcp`).

In [ ]:
#@title 1.1 Deploy MCP Proxy for Loans, API Product & Developer App via AFT
# 1) Ensures the OAuth 2.0 Token Server (features/auth-oauth-server.yaml) is active
# 2) Deploys templates/mcp-gateway-loan-product.yaml via AFT and waits until READY
# 3) Deploys config/products/ai-loan-agent-package.yaml via AFT
# 4) Deploys config/apps/user_apps_mcp_loan.yaml via AFT

LOAN_PROXY_NAME = "mcp-gateway-loan-product"
OAUTH_PROXY_NAME = "auth-oauth-server"

LOAN_TEMPLATE_PATH = "templates/mcp-gateway-loan-product.yaml"
LOAN_PRODUCT_PATH = "config/products/ai-loan-agent-package.yaml"
LOAN_APP_PATH = "config/apps/user_apps_mcp_loan.yaml"
OAUTH_SERVER_PATH = "features/auth-oauth-server.yaml"

ensure_repo_files([
    LOAN_TEMPLATE_PATH,
    "features/mcp-loan-endpoint.yaml",
    "features/api-quota-policy-spike-oauth.yaml",
    OAUTH_SERVER_PATH,
    LOAN_PRODUCT_PATH,
    LOAN_APP_PATH,
])

# 0. Ensure OAuth 2.0 Token Server is deployed so /v1/oauth/token can mint tokens
oauth_status, oauth_deployments = gcp_request(
    "GET", f"{APIGEE_BASE}/environments/{APIGEE_ENV}/apis/{OAUTH_PROXY_NAME}/deployments"
)
if oauth_status != 200 or not oauth_deployments.get("deployments"):
    run_aft_deploy(OAUTH_SERVER_PATH, f"[0/3] Deploying OAuth 2.0 Token Server ({OAUTH_PROXY_NAME})")
wait_for_proxy_ready(OAUTH_PROXY_NAME, APIGEE_ENV)

# 1. Deploy the MCP Loans Proxy
run_aft_deploy(LOAN_TEMPLATE_PATH, "[1/3] Deploying MCP Loans Proxy")
loan_revision = wait_for_proxy_ready(LOAN_PROXY_NAME, APIGEE_ENV)

# 2. Deploy the Loans API Product / Package
ensure_env_in_product_yaml(LOAN_PRODUCT_PATH, APIGEE_ENV)
run_aft_deploy(LOAN_PRODUCT_PATH, "[2/3] Deploying Loans API Product / Package")
print(f"✅ Deployed API Product: {LOAN_PRODUCT_PATH}")

# 3. Deploy the Loans Developer & App
run_aft_deploy(LOAN_APP_PATH, "[3/3] Deploying Loans Developer & App")
print(f"✅ Deployed Developer & App: {LOAN_APP_PATH}")

OAUTH_TOKEN_ENDPOINT = f"https://{HOSTNAME}/v1/oauth/token"
MCP_LOAN_ENDPOINT = f"https://{HOSTNAME}/loanservice/mcp"
os.environ.update({
    "PROXY_NAME_MCP_LOAN": LOAN_PROXY_NAME,
    "OAUTH_TOKEN_ENDPOINT": OAUTH_TOKEN_ENDPOINT,
    "MCP_LOAN_ENDPOINT": MCP_LOAN_ENDPOINT,
})

print("=" * 72)
print(f"✅ Deployed MCP Proxy : {LOAN_PROXY_NAME} (rev={loan_revision})")
print(f"✅ API Product        : ai-loan-agent-package")
print(f"✅ Developer App      : Loan Agent App (loan-agent@example.com)")
print(f"✅ OAuth Token URL    : {OAUTH_TOKEN_ENDPOINT}")
print(f"✅ MCP Loan Endpoint  : {MCP_LOAN_ENDPOINT}")
print("=" * 72)

## Step 1.2 — Test the Loans MCP Gateway with the Interactive Web Client

### What This Step Does
Launches an interactive **Loans MCP Web Client** directly inside Google Colab. Under the hood, the client performs the standard Model Context Protocol (MCP) handshake over HTTPS:
1. **Mints an OAuth 2.0 Access Token** from `/v1/oauth/token` using your app's `CLIENT_ID` and `CLIENT_SECRET`.
2. **Initializes an MCP Session** (`initialize` + `notifications/initialized`) with the Apigee Loans MCP proxy at `/loanservice/mcp`.
3. **Dynamically Discovers Tools** (`tools/list`) and builds an interactive input form from the server's live JSON Schema—no hardcoded tool names or guessed arguments.
4. **Executes Tool Calls** (`tools/call`) with full UTF-8 character encoding support so international names, accented text, and currency symbols (`€`, `$`, `£`) render cleanly.

### Why It Is Necessary
Verifies end-to-end that Apigee is properly authenticating OAuth 2.0 tokens, inspecting MCP JSON-RPC 2.0 traffic, and streaming responses back from the backend Loans MCP server.

### Inputs & Outputs
- **What You Need to Do (Inputs):**
  1. Keep the default `CLIENT_ID` and `CLIENT_SECRET` (or enter custom credentials) and run the cell.
  2. In the web client UI, choose a tool from the **Select MCP Tool** dropdown, fill in the dynamically generated form fields, and click **`▶ Execute Tool (tools/call)`**.
- **What You Will Get (Outputs):**
  - A live **Connection & Handshake Banner** showing each protocol step and its latency in milliseconds.
  - A **Formatted Tool Output** card displaying the loan calculation or customer result, plus an expandable **JSON-RPC 2.0 Protocol Inspector** showing the exact request and response payloads.

In [ ]:
#@title 1.2 Test MCP Loan Proxy (Standard MCP Protocol & Interactive Web Client)
#@markdown ### OAuth 2.0 App Credentials (`Loan Agent App`)
CLIENT_ID = "loan-agent-app-key-123" #@param {type:"string"}
CLIENT_SECRET = "loan-agent-app-secret-123" #@param {type:"string"}

from dataclasses import dataclass
import html
import json
import time
from typing import Any, Dict, List, Optional, Tuple
from IPython.display import HTML, display
import ipywidgets as widgets
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


# ============================================================================
# Modular UTF-8 MCP Streamable HTTP Client & Web UI Components
# ============================================================================
@dataclass
class HandshakeMetric:
    """Stores telemetry for a single step of the MCP handshake."""
    step_name: str
    status_code: int
    latency_ms: int
    detail: str
    ok: bool = True


@dataclass
class ToolCallOutcome:
    """Stores the full request/response context of an MCP tools/call invocation."""
    tool_name: str
    request_headers: Dict[str, str]
    request_payload: Dict[str, Any]
    status_code: int
    response_headers: Dict[str, str]
    response_body: Dict[str, Any]
    latency_ms: int
    session_id: Optional[str]
    reinitialized: bool = False


class MCPStreamableClient:
    """Spec-compliant Model Context Protocol (MCP) Streamable HTTP client with strict UTF-8 handling."""

    PROTOCOL_VERSION = "2024-11-05"

    def __init__(
        self,
        mcp_url: str,
        oauth_url: str,
        client_id: str,
        client_secret: str,
        auth_mode: str = "oauth2",
    ) -> None:
        self.mcp_url = mcp_url
        self.oauth_url = oauth_url
        self.client_id = client_id.strip()
        self.client_secret = client_secret.strip()
        self.auth_mode = auth_mode  # "oauth2" or "x-api-key"

        self.access_token: Optional[str] = None
        self.token_expires_in: Optional[Any] = None
        self.session_id: Optional[str] = None
        self.protocol_version: str = self.PROTOCOL_VERSION
        self.server_info: Dict[str, Any] = {}
        self.server_capabilities: Dict[str, Any] = {}
        self.tools: List[Dict[str, Any]] = []
        self.handshake_metrics: List[HandshakeMetric] = []

        self._rpc_id = 0
        self._http = requests.Session()
        self._http.verify = False

    def _next_id(self) -> int:
        self._rpc_id += 1
        return self._rpc_id

    @property
    def masked_credential(self) -> str:
        """Return a redacted representation of the active token or API key."""
        secret = (
            self.access_token
            if self.auth_mode == "oauth2" and self.access_token
            else self.client_id
        )
        if not secret:
            return "none"
        return "***" if len(secret) <= 12 else f"{secret[:8]}...{secret[-4:]}"

    def build_headers(
        self,
        include_session: bool = True,
        mask_sensitive: bool = False,
    ) -> Dict[str, str]:
        """Construct HTTP headers with explicit UTF-8 charset negotiation."""
        headers: Dict[str, str] = {
            "Content-Type": "application/json; charset=utf-8",
            "Accept": "application/json; charset=utf-8, text/event-stream; charset=utf-8",
            "Accept-Charset": "utf-8",
        }
        if self.auth_mode == "oauth2" and self.access_token:
            token_val = self.masked_credential if mask_sensitive else self.access_token
            headers["Authorization"] = f"Bearer {token_val}"
        else:
            key_val = self.masked_credential if mask_sensitive else self.client_id
            headers["x-api-key"] = key_val

        if include_session and self.session_id:
            headers["mcp-session-id"] = self.session_id
            headers["mcp-protocol-version"] = self.protocol_version
        return headers

    @staticmethod
    def decode_utf8_response(resp: requests.Response) -> Dict[str, Any]:
        """Decode HTTP JSON or SSE response body strictly as UTF-8."""
        resp.encoding = "utf-8"
        raw_text = resp.content.decode("utf-8", errors="replace").strip()
        if not raw_text:
            return {}

        content_type = (resp.headers.get("Content-Type") or "").lower()
        if "text/event-stream" in content_type or raw_text.startswith(("event:", "data:")):
            last_event: Dict[str, Any] = {}
            for line in raw_text.splitlines():
                stripped = line.strip()
                if stripped.startswith("data:") and stripped[5:].strip():
                    try:
                        last_event = json.loads(stripped[5:].strip())
                    except ValueError:
                        pass
            if last_event:
                return last_event

        try:
            return json.loads(raw_text)
        except ValueError:
            return {"raw_text": raw_text}

    def _post_json_utf8(
        self,
        payload: Dict[str, Any],
        include_session: bool = True,
        timeout: int = 25,
    ) -> Tuple[requests.Response, int]:
        """Send a UTF-8 encoded JSON-RPC payload and return (response, latency_ms)."""
        body_bytes = json.dumps(payload, ensure_ascii=False).encode("utf-8")
        start_time = time.perf_counter()
        resp = self._http.post(
            self.mcp_url,
            headers=self.build_headers(include_session=include_session),
            data=body_bytes,
            timeout=timeout,
        )
        resp.encoding = "utf-8"
        latency_ms = int((time.perf_counter() - start_time) * 1000)
        for header_name, header_val in resp.headers.items():
            if header_name.lower() == "mcp-session-id" and header_val:
                self.session_id = header_val.strip()
                break
        return resp, latency_ms

    def mint_oauth_token(self) -> HandshakeMetric:
        """Mint an OAuth 2.0 client_credentials token from Apigee."""
        start_time = time.perf_counter()
        resp = self._http.post(
            self.oauth_url,
            auth=(self.client_id, self.client_secret),
            headers={
                "Content-Type": "application/x-www-form-urlencoded; charset=utf-8",
                "Accept": "application/json; charset=utf-8",
            },
            data={"grant_type": "client_credentials"},
            timeout=20,
        )
        resp.encoding = "utf-8"
        latency_ms = int((time.perf_counter() - start_time) * 1000)
        if resp.status_code != 200:
            raise RuntimeError(
                f"OAuth 2.0 token request failed (HTTP {resp.status_code}, {latency_ms}ms): "
                f"{resp.content.decode('utf-8', errors='replace')[:400]}"
            )
        token_data = json.loads(resp.content.decode("utf-8", errors="replace"))
        self.access_token = token_data.get("access_token", "")
        self.token_expires_in = token_data.get("expires_in", "N/A")
        self.auth_mode = "oauth2"
        return HandshakeMetric(
            "1. OAuth 2.0 Token",
            resp.status_code,
            latency_ms,
            f"token={self.masked_credential} (TTL {self.token_expires_in}s)",
        )

    def initialize_session(self) -> Tuple[HandshakeMetric, HandshakeMetric]:
        """Run MCP 'initialize' and 'notifications/initialized' handshake."""
        self.session_id = None
        init_payload = {
            "jsonrpc": "2.0",
            "id": self._next_id(),
            "method": "initialize",
            "params": {
                "protocolVersion": self.PROTOCOL_VERSION,
                "capabilities": {},
                "clientInfo": {"name": "apigee-colab-mcp-client", "version": "2.0.0"},
            },
        }
        resp_init, ms_init = self._post_json_utf8(init_payload, include_session=False)
        body_init = self.decode_utf8_response(resp_init)
        if resp_init.status_code not in (200, 202) or "error" in body_init:
            raise RuntimeError(
                f"MCP initialize failed (HTTP {resp_init.status_code}): "
                f"{json.dumps(body_init, ensure_ascii=False)[:400]}"
            )

        res_obj = body_init.get("result", {}) if isinstance(body_init, dict) else {}
        self.protocol_version = res_obj.get("protocolVersion", self.PROTOCOL_VERSION)
        self.server_info = res_obj.get("serverInfo", {})
        self.server_capabilities = res_obj.get("capabilities", {})

        srv_name = self.server_info.get("name", "mcp-server")
        srv_ver = self.server_info.get("version", "")
        m_init = HandshakeMetric(
            "2. MCP initialize",
            resp_init.status_code,
            ms_init,
            f"server={srv_name} {srv_ver} | proto={self.protocol_version}".strip(),
        )

        notif_payload = {"jsonrpc": "2.0", "method": "notifications/initialized"}
        resp_notif, ms_notif = self._post_json_utf8(
            notif_payload, include_session=True, timeout=15
        )
        m_notif = HandshakeMetric(
            "3. notifications/initialized",
            resp_notif.status_code,
            ms_notif,
            f"session={self.session_id or 'stateless'}",
        )
        return m_init, m_notif

    def list_tools(self) -> Tuple[HandshakeMetric, List[Dict[str, Any]]]:
        """Discover available MCP tools dynamically via 'tools/list'."""
        payload = {"jsonrpc": "2.0", "id": self._next_id(), "method": "tools/list"}
        resp, latency_ms = self._post_json_utf8(payload, include_session=True)
        body = self.decode_utf8_response(resp)

        if resp.status_code == 200 and isinstance(body, dict) and "error" in body:
            resp_retry, ms_retry = self._post_json_utf8(
                {**payload, "id": self._next_id(), "params": {}}, include_session=True
            )
            body_retry = self.decode_utf8_response(resp_retry)
            if "error" not in body_retry:
                resp, latency_ms, body = resp_retry, ms_retry, body_retry

        if resp.status_code != 200 or (isinstance(body, dict) and "error" in body):
            raise RuntimeError(
                f"MCP tools/list failed (HTTP {resp.status_code}): "
                f"{json.dumps(body, ensure_ascii=False)[:400]}"
            )

        tools = (
            body.get("result", {}).get("tools", [])
            if isinstance(body, dict)
            else []
        )
        self.tools = tools
        metric = HandshakeMetric(
            "4. MCP tools/list",
            resp.status_code,
            latency_ms,
            f"discovered {len(tools)} tool(s)",
        )
        return metric, tools

    def bootstrap(self, refresh_auth: bool = False) -> List[Dict[str, Any]]:
        """Execute the full authentication + MCP discovery lifecycle."""
        metrics: List[HandshakeMetric] = []
        if self.auth_mode == "oauth2":
            if refresh_auth or not self.access_token:
                metrics.append(self.mint_oauth_token())
            else:
                metrics.append(
                    HandshakeMetric(
                        "1. OAuth 2.0 Token",
                        200,
                        0,
                        f"reused token={self.masked_credential}",
                    )
                )
        else:
            if refresh_auth:
                metrics.append(self.mint_oauth_token())
            else:
                metrics.append(
                    HandshakeMetric(
                        "1. App Auth (x-api-key)",
                        200,
                        0,
                        f"key={self.masked_credential}",
                    )
                )

        try:
            m_init, m_notif = self.initialize_session()
        except RuntimeError:
            if self.auth_mode == "x-api-key" and self.client_secret:
                metrics = [self.mint_oauth_token()]
                m_init, m_notif = self.initialize_session()
            else:
                raise

        m_tools, tools = self.list_tools()
        metrics.extend([m_init, m_notif, m_tools])
        self.handshake_metrics = metrics
        return tools

    def call_tool(self, tool_name: str, arguments: Dict[str, Any]) -> ToolCallOutcome:
        """Invoke an MCP tool via 'tools/call', auto-recovering if session expired."""
        if not self.session_id:
            self.initialize_session()

        def _send_call() -> Tuple[requests.Response, int, Dict[str, Any], Dict[str, Any]]:
            payload = {
                "jsonrpc": "2.0",
                "id": self._next_id(),
                "method": "tools/call",
                "params": {"name": tool_name, "arguments": arguments},
            }
            resp, ms = self._post_json_utf8(payload, include_session=True, timeout=30)
            return resp, ms, payload, self.decode_utf8_response(resp)

        resp, latency_ms, req_payload, body = _send_call()
        reinitialized = False
        body_str = json.dumps(body, ensure_ascii=False).lower()
        if resp.status_code in (400, 404) or "session" in body_str:
            self.initialize_session()
            resp, latency_ms, req_payload, body = _send_call()
            reinitialized = True

        return ToolCallOutcome(
            tool_name=tool_name,
            request_headers=self.build_headers(include_session=True, mask_sensitive=True),
            request_payload=req_payload,
            status_code=resp.status_code,
            response_headers=dict(resp.headers),
            response_body=body,
            latency_ms=latency_ms,
            session_id=self.session_id,
            reinitialized=reinitialized,
        )


class MCPWorkbenchUI:
    """Modern, UTF-8 safe interactive web client for exploring and testing MCP servers in Colab."""

    def __init__(
        self,
        client: MCPStreamableClient,
        title: str,
        subtitle: str,
        proxy_name: str,
        active_revision: str,
        accent_color: str = "#2563eb",
        accent_dark: str = "#1e3a8a",
    ) -> None:
        self.client = client
        self.title = title
        self.subtitle = subtitle
        self.proxy_name = proxy_name
        self.active_revision = active_revision
        self.accent_color = accent_color
        self.accent_dark = accent_dark

        self.tools_by_name: Dict[str, Dict[str, Any]] = {
            t["name"]: t for t in self.client.tools if t.get("name")
        }
        self._param_widgets: Dict[str, Tuple[widgets.Widget, Dict[str, Any], bool]] = {}

        self._build_widgets()

    @staticmethod
    def _resolve_json_type(prop_spec: Dict[str, Any]) -> str:
        if "type" in prop_spec and isinstance(prop_spec["type"], str):
            return prop_spec["type"]
        for union_key in ("anyOf", "oneOf"):
            if union_key in prop_spec and isinstance(prop_spec[union_key], list):
                for option in prop_spec[union_key]:
                    if isinstance(option, dict) and option.get("type") not in (None, "null"):
                        return str(option["type"])
        return "string"

    def _base_styles(self) -> str:
        return f"""
        <meta charset="utf-8">
        <style>
          .mcp-ui-wrap {{
            font-family: 'Inter', -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
            color: #0f172a;
            max-width: 980px;
            line-height: 1.5;
            box-sizing: border-box;
          }}
          .mcp-ui-wrap code, .mcp-ui-wrap pre {{
            font-family: 'JetBrains Mono', 'Fira Code', ui-monospace, SFMono-Regular, Menlo, monospace;
          }}
          .mcp-banner {{
            background: linear-gradient(135deg, #0f172a 0%, {self.accent_dark} 100%);
            color: #f8fafc;
            padding: 20px 24px;
            border-radius: 14px;
            box-shadow: 0 10px 25px -5px rgba(15, 23, 42, 0.22);
            margin-bottom: 14px;
          }}
          .mcp-banner-top {{
            display: flex;
            justify-content: space-between;
            align-items: flex-start;
            flex-wrap: wrap;
            gap: 12px;
          }}
          .mcp-banner-title {{
            font-size: 19px;
            font-weight: 700;
            letter-spacing: -0.01em;
            margin: 0;
          }}
          .mcp-banner-sub {{
            font-size: 13px;
            color: #cbd5e1;
            margin-top: 4px;
          }}
          .mcp-status-pill {{
            background: rgba(16, 185, 129, 0.18);
            border: 1px solid rgba(52, 211, 153, 0.45);
            color: #6ee7b7;
            padding: 5px 12px;
            border-radius: 999px;
            font-size: 12px;
            font-weight: 600;
          }}
          .mcp-meta-grid {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
            gap: 10px;
            margin-top: 16px;
            padding-top: 14px;
            border-top: 1px solid rgba(255, 255, 255, 0.14);
            font-size: 12px;
          }}
          .mcp-meta-box {{
            background: rgba(255, 255, 255, 0.06);
            padding: 8px 12px;
            border-radius: 8px;
          }}
          .mcp-meta-label {{
            color: #94a3b8;
            font-size: 10.5px;
            text-transform: uppercase;
            letter-spacing: 0.05em;
            display: block;
          }}
          .mcp-meta-val {{
            color: #f8fafc;
            font-weight: 600;
            word-break: break-all;
          }}
          .mcp-steps-row {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(210px, 1fr));
            gap: 8px;
            margin-top: 12px;
          }}
          .mcp-step-chip {{
            background: rgba(15, 23, 42, 0.48);
            border: 1px solid rgba(148, 163, 184, 0.24);
            border-radius: 8px;
            padding: 8px 11px;
            font-size: 11.5px;
          }}
          .mcp-step-head {{
            display: flex;
            justify-content: space-between;
            font-weight: 600;
            color: #e2e8f0;
          }}
          .mcp-step-sub {{
            color: #94a3b8;
            font-size: 11px;
            margin-top: 3px;
            white-space: nowrap;
            overflow: hidden;
            text-overflow: ellipsis;
          }}
          .mcp-section-card {{
            background: #ffffff;
            border: 1px solid #e2e8f0;
            border-radius: 12px;
            padding: 14px 18px;
            margin-bottom: 10px;
            box-shadow: 0 1px 3px rgba(15, 23, 42, 0.04);
          }}
          .mcp-badge {{
            display: inline-block;
            padding: 2px 8px;
            border-radius: 6px;
            font-size: 11px;
            font-weight: 600;
            margin-left: 6px;
          }}
          .mcp-badge-type {{
            background: #eff6ff;
            color: {self.accent_color};
            border: 1px solid #bfdbfe;
          }}
          .mcp-badge-req {{
            background: #fef2f2;
            color: #dc2626;
            border: 1px solid #fecaca;
          }}
          .mcp-badge-opt {{
            background: #f8fafc;
            color: #64748b;
            border: 1px solid #e2e8f0;
          }}
          .mcp-result-card {{
            background: #ffffff;
            border: 1px solid #cbd5e1;
            border-radius: 12px;
            overflow: hidden;
            margin-top: 10px;
            box-shadow: 0 4px 12px rgba(15, 23, 42, 0.05);
          }}
          .mcp-result-header {{
            padding: 12px 18px;
            display: flex;
            justify-content: space-between;
            align-items: center;
            border-bottom: 1px solid #e2e8f0;
          }}
          .mcp-pre-light {{
            background: #f8fafc;
            color: #0f172a;
            border: 1px solid #e2e8f0;
            border-radius: 8px;
            padding: 12px 14px;
            font-size: 12.5px;
            overflow-x: auto;
            white-space: pre-wrap;
            word-break: break-word;
            margin: 6px 0 0 0;
          }}
          .mcp-pre-dark {{
            background: #0f172a;
            color: #e2e8f0;
            border-radius: 8px;
            padding: 12px 14px;
            font-size: 12px;
            overflow-x: auto;
            white-space: pre-wrap;
            word-break: break-word;
            margin: 6px 0 0 0;
          }}
        </style>
        """

    def _render_banner_html(self) -> str:
        srv_name = html.escape(str(self.client.server_info.get("name", "mcp-server")))
        srv_ver = html.escape(str(self.client.server_info.get("version", "1.0.0")))
        sess_id = html.escape(str(self.client.session_id or "stateless"))
        auth_badge = (
            f"OAuth 2.0 Bearer ({html.escape(self.client.masked_credential)})"
            if self.client.auth_mode == "oauth2"
            else f"API Key ({html.escape(self.client.masked_credential)})"
        )

        step_chips = ""
        for metric in self.client.handshake_metrics:
            step_chips += f"""
            <div class="mcp-step-chip">
              <div class="mcp-step-head">
                <span>✅ {html.escape(metric.step_name)}</span>
                <span style="color:#6ee7b7;">HTTP {metric.status_code} · {metric.latency_ms}ms</span>
              </div>
              <div class="mcp-step-sub" title="{html.escape(metric.detail)}">{html.escape(metric.detail)}</div>
            </div>
            """

        return f"""
        {self._base_styles()}
        <div class="mcp-ui-wrap">
          <div class="mcp-banner">
            <div class="mcp-banner-top">
              <div>
                <h3 class="mcp-banner-title">{html.escape(self.title)}</h3>
                <div class="mcp-banner-sub">{html.escape(self.subtitle)}</div>
              </div>
              <div class="mcp-status-pill">● MCP SESSION ACTIVE ({len(self.client.tools)} TOOLS)</div>
            </div>
            <div class="mcp-meta-grid">
              <div class="mcp-meta-box">
                <span class="mcp-meta-label">Apigee Proxy & Revision</span>
                <span class="mcp-meta-val">{html.escape(self.proxy_name)} (rev {html.escape(self.active_revision)})</span>
              </div>
              <div class="mcp-meta-box">
                <span class="mcp-meta-label">MCP Server & Protocol</span>
                <span class="mcp-meta-val">{srv_name} v{srv_ver} ({html.escape(self.client.protocol_version)})</span>
              </div>
              <div class="mcp-meta-box">
                <span class="mcp-meta-label">Authentication & Session ID</span>
                <span class="mcp-meta-val">{auth_badge} · {sess_id}</span>
              </div>
              <div class="mcp-meta-box">
                <span class="mcp-meta-label">Gateway Endpoint (UTF-8)</span>
                <span class="mcp-meta-val">{html.escape(self.client.mcp_url)}</span>
              </div>
            </div>
            <div class="mcp-steps-row">{step_chips}</div>
          </div>
        </div>
        """

    def _render_tool_header_html(self, tool_spec: Dict[str, Any]) -> str:
        t_name = html.escape(tool_spec.get("name", ""))
        t_desc = html.escape(tool_spec.get("description", "No description provided."))
        schema = tool_spec.get("inputSchema") or {"type": "object", "properties": {}}
        prop_count = len(schema.get("properties") or {})
        req_count = len(schema.get("required") or [])
        schema_str = html.escape(json.dumps(schema, indent=2, ensure_ascii=False))

        return f"""
        {self._base_styles()}
        <div class="mcp-ui-wrap">
          <div class="mcp-section-card">
            <div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:8px;">
              <div>
                <span style="font-size:11px;font-weight:700;text-transform:uppercase;color:{self.accent_color};letter-spacing:0.05em;">
                  Step 2 · Configure Tool Parameters
                </span>
                <div style="font-size:15px;font-weight:700;color:#0f172a;margin-top:2px;">
                  <code>{t_name}</code>
                </div>
              </div>
              <span style="font-size:12px;color:#475569;background:#f1f5f9;padding:4px 10px;border-radius:999px;font-weight:600;">
                {prop_count} parameter(s) · {req_count} required
              </span>
            </div>
            <div style="font-size:13px;color:#334155;margin-top:6px;">{t_desc}</div>
            <details style="margin-top:8px;font-size:12px;color:#475569;">
              <summary style="cursor:pointer;font-weight:600;color:{self.accent_color};">
                Inspect Discovered JSON Schema (inputSchema)
              </summary>
              <pre class="mcp-pre-light">{schema_str}</pre>
            </details>
          </div>
        </div>
        """

    def _render_call_outcome_html(self, outcome: ToolCallOutcome) -> str:
        body = outcome.response_body
        is_rpc_err = isinstance(body, dict) and "error" in body
        is_tool_err = (
            isinstance(body, dict)
            and isinstance(body.get("result"), dict)
            and bool(body["result"].get("isError"))
        )
        passed = outcome.status_code == 200 and not is_rpc_err and not is_tool_err

        bg_header = "#ecfdf5" if passed else "#fef2f2"
        fg_header = "#065f46" if passed else "#991b1b"
        status_icon = "✅" if passed else "⚠️"
        status_label = (
            "MCP TOOL EXECUTION SUCCEEDED"
            if passed
            else "MCP GATEWAY / SERVER RETURNED AN ERROR"
        )

        blocks_html = ""
        if isinstance(body, dict) and isinstance(body.get("result"), dict):
            content_list = body["result"].get("content", [])
            if isinstance(content_list, list) and content_list:
                rendered_items = []
                for item in content_list:
                    if isinstance(item, dict) and item.get("type") == "text":
                        raw_txt = str(item.get("text", ""))
                        try:
                            parsed_json = json.loads(raw_txt)
                            formatted_txt = json.dumps(parsed_json, indent=2, ensure_ascii=False)
                        except ValueError:
                            formatted_txt = raw_txt
                        rendered_items.append(
                            f"<pre class='mcp-pre-light'>{html.escape(formatted_txt)}</pre>"
                        )
                    else:
                        rendered_items.append(
                            f"<pre class='mcp-pre-light'>{html.escape(json.dumps(item, indent=2, ensure_ascii=False))}</pre>"
                        )
                blocks_html = "".join(rendered_items)
            elif "structuredContent" in body["result"]:
                sc_str = json.dumps(
                    body["result"]["structuredContent"], indent=2, ensure_ascii=False
                )
                blocks_html = f"<pre class='mcp-pre-light'>{html.escape(sc_str)}</pre>"
        elif is_rpc_err:
            err_str = json.dumps(body.get("error"), indent=2, ensure_ascii=False)
            blocks_html = (
                f"<pre class='mcp-pre-light' style='border-color:#fca5a5;color:#991b1b;'>"
                f"{html.escape(err_str)}</pre>"
            )

        if not blocks_html:
            fallback_str = json.dumps(body, indent=2, ensure_ascii=False)
            blocks_html = f"<pre class='mcp-pre-light'>{html.escape(fallback_str)}</pre>"

        req_dump = html.escape(
            json.dumps(
                {"headers": outcome.request_headers, "body": outcome.request_payload},
                indent=2,
                ensure_ascii=False,
            )
        )
        resp_dump = html.escape(json.dumps(body, indent=2, ensure_ascii=False))

        return f"""
        {self._base_styles()}
        <div class="mcp-ui-wrap">
          <div class="mcp-result-card">
            <div class="mcp-result-header" style="background:{bg_header};color:{fg_header};">
              <div style="font-weight:700;font-size:13.5px;">
                {status_icon} {status_label} · <code>{html.escape(outcome.tool_name)}</code>
              </div>
              <div style="font-size:12px;font-weight:600;">
                HTTP {outcome.status_code} · {outcome.latency_ms} ms · UTF-8 Verified
              </div>
            </div>
            <div style="padding:16px 18px;">
              <div style="font-size:11.5px;font-weight:700;text-transform:uppercase;color:#475569;letter-spacing:0.05em;">
                Formatted Tool Output
              </div>
              {blocks_html}
              <details style="margin-top:14px;">
                <summary style="cursor:pointer;font-size:12px;font-weight:600;color:{self.accent_color};">
                  Inspect Raw JSON-RPC 2.0 Request & Response Payloads
                </summary>
                <div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(320px,1fr));gap:12px;margin-top:8px;">
                  <div>
                    <div style="font-size:11px;font-weight:700;color:#475569;">📤 JSON-RPC 2.0 Request</div>
                    <pre class="mcp-pre-dark">{req_dump}</pre>
                  </div>
                  <div>
                    <div style="font-size:11px;font-weight:700;color:#475569;">📥 JSON-RPC 2.0 Response</div>
                    <pre class="mcp-pre-dark">{resp_dump}</pre>
                  </div>
                </div>
              </details>
            </div>
          </div>
        </div>
        """

    def _build_widgets(self) -> None:
        self.banner_widget = widgets.HTML(self._render_banner_html())
        tool_options = [
            (f"🔧 {t['name']}", t["name"]) for t in self.client.tools if t.get("name")
        ]
        self.tool_dropdown = widgets.Dropdown(
            options=tool_options or [("No tools discovered", "")],
            value=tool_options[0][1] if tool_options else "",
            description="1. Tool:",
            style={"description_width": "70px"},
            layout=widgets.Layout(width="440px"),
        )
        self.refresh_btn = widgets.Button(
            description="🔄 Refresh Tools",
            button_style="info",
            layout=widgets.Layout(width="160px", height="34px"),
        )
        auth_btn_label = (
            "🔑 Re-mint OAuth Token"
            if self.client.auth_mode == "oauth2"
            else "🔑 Mint OAuth Token"
        )
        self.auth_btn = widgets.Button(
            description=auth_btn_label,
            button_style="",
            layout=widgets.Layout(width="185px", height="34px"),
        )
        self.tool_header_widget = widgets.HTML()
        self.params_container = widgets.VBox(layout=widgets.Layout(width="100%", max_width="980px"))
        self.execute_btn = widgets.Button(
            description="▶ Execute Tool (tools/call)",
            button_style="primary",
            layout=widgets.Layout(width="240px", height="38px"),
        )
        self.clear_btn = widgets.Button(
            description="🧹 Clear Output",
            button_style="",
            layout=widgets.Layout(width="140px", height="38px"),
        )
        self.output_area = widgets.Output(layout=widgets.Layout(width="100%", max_width="980px"))

        self.tool_dropdown.observe(self._on_tool_selected, names="value")
        self.refresh_btn.on_click(lambda _: self._sync_session(refresh_auth=False))
        self.auth_btn.on_click(lambda _: self._sync_session(refresh_auth=True))
        self.execute_btn.on_click(self._on_execute_clicked)
        self.clear_btn.on_click(lambda _: self.output_area.clear_output())

        self._rebuild_form(self.tool_dropdown.value)

    def _rebuild_form(self, tool_name: str) -> None:
        self._param_widgets.clear()
        tool_spec = self.tools_by_name.get(tool_name)
        if not tool_spec:
            self.tool_header_widget.value = (
                f"{self._base_styles()}<div class='mcp-ui-wrap'>"
                "<div class='mcp-section-card'>⚠️ No MCP tool selected.</div></div>"
            )
            self.params_container.children = []
            return

        self.tool_header_widget.value = self._render_tool_header_html(tool_spec)
        schema = tool_spec.get("inputSchema") or {}
        properties = schema.get("properties") or {}
        required_set = set(schema.get("required") or [])

        if not properties:
            self.params_container.children = [
                widgets.HTML(
                    f"{self._base_styles()}<div class='mcp-ui-wrap'><div class='mcp-section-card' "
                    "style='color:#475569;font-size:13px;'>ℹ️ This tool takes no input parameters. "
                    "Click <b>▶ Execute Tool</b> below.</div></div>"
                )
            ]
            return

        rows = []
        for param_name, prop_spec in properties.items():
            param_type = self._resolve_json_type(prop_spec)
            is_required = param_name in required_set
            desc = prop_spec.get("description", "")
            default_val = prop_spec.get("default")
            enum_vals = prop_spec.get("enum")
            ctrl_layout = widgets.Layout(width="360px")

            if enum_vals and isinstance(enum_vals, list):
                ctrl = widgets.Dropdown(
                    options=[(str(v), v) for v in enum_vals],
                    value=default_val if default_val in enum_vals else enum_vals[0],
                    layout=ctrl_layout,
                )
            elif param_type == "boolean":
                ctrl = widgets.Checkbox(
                    value=bool(default_val) if default_val is not None else False,
                    indent=False,
                    layout=ctrl_layout,
                )
            elif param_type == "integer":
                ctrl = widgets.IntText(
                    value=int(default_val) if isinstance(default_val, (int, float)) else 0,
                    layout=ctrl_layout,
                )
            elif param_type == "number":
                ctrl = widgets.FloatText(
                    value=float(default_val) if isinstance(default_val, (int, float)) else 0.0,
                    step=0.01,
                    layout=ctrl_layout,
                )
            elif param_type in ("array", "object"):
                fallback_json = (
                    json.dumps(default_val, ensure_ascii=False)
                    if default_val is not None
                    else ("[]" if param_type == "array" else "{}")
                )
                ctrl = widgets.Textarea(
                    value=fallback_json,
                    placeholder=f"UTF-8 JSON {param_type}",
                    layout=widgets.Layout(width="360px", height="64px"),
                )
            else:
                ctrl = widgets.Text(
                    value=str(default_val) if default_val is not None else "",
                    placeholder=desc or f"Enter {param_name}",
                    layout=ctrl_layout,
                )

            self._param_widgets[param_name] = (ctrl, prop_spec, is_required)
            req_cls = "mcp-badge-req" if is_required else "mcp-badge-opt"
            req_txt = "required" if is_required else "optional"
            label_html = widgets.HTML(
                f"{self._base_styles()}<div class='mcp-ui-wrap' style='width:540px;'>"
                f"<div><code style='font-size:13px;font-weight:700;color:#0f172a;'>"
                f"{html.escape(param_name)}</code>"
                f"<span class='mcp-badge mcp-badge-type'>{html.escape(param_type)}</span>"
                f"<span class='mcp-badge {req_cls}'>{req_txt}</span></div>"
                f"<div style='font-size:12px;color:#64748b;margin-top:2px;'>"
                f"{html.escape(desc)}</div></div>"
            )
            rows.append(
                widgets.HBox(
                    [label_html, ctrl],
                    layout=widgets.Layout(
                        align_items="center",
                        justify_content="space-between",
                        padding="8px 14px",
                        border="1px solid #e2e8f0",
                        margin="0 0 6px 0",
                        width="100%",
                    ),
                )
            )
        self.params_container.children = rows

    def _collect_arguments(self) -> Tuple[str, Dict[str, Any]]:
        tool_name = self.tool_dropdown.value
        if not tool_name:
            raise ValueError("Please select an MCP tool first.")
        arguments: Dict[str, Any] = {}
        for param_name, (ctrl, prop_spec, is_required) in self._param_widgets.items():
            param_type = self._resolve_json_type(prop_spec)
            raw_val = ctrl.value
            if param_type in ("array", "object"):
                text_val = str(raw_val).strip()
                if not text_val and not is_required:
                    continue
                arguments[param_name] = json.loads(text_val)
            elif param_type == "string":
                if raw_val == "" and not is_required:
                    continue
                arguments[param_name] = str(raw_val)
            elif param_type == "integer":
                arguments[param_name] = int(raw_val)
            elif param_type == "number":
                arguments[param_name] = float(raw_val)
            elif param_type == "boolean":
                arguments[param_name] = bool(raw_val)
            else:
                arguments[param_name] = raw_val
        return tool_name, arguments

    def _on_tool_selected(self, change: Dict[str, Any]) -> None:
        if change.get("name") == "value":
            self._rebuild_form(change.get("new"))

    def _sync_session(self, refresh_auth: bool) -> None:
        with self.output_area:
            self.output_area.clear_output(wait=True)
            display(
                HTML(
                    f"{self._base_styles()}<div class='mcp-ui-wrap' style='font-size:13px;"
                    f"color:{self.accent_color};font-weight:600;'>"
                    "⏳ Synchronizing MCP session and running <code>tools/list</code>...</div>"
                )
            )
            try:
                discovered = self.client.bootstrap(refresh_auth=refresh_auth)
                self.tools_by_name = {t["name"]: t for t in discovered if t.get("name")}
                self.banner_widget.value = self._render_banner_html()
                options = [(f"🔧 {t['name']}", t["name"]) for t in discovered if t.get("name")]
                self.tool_dropdown.options = options
                if options:
                    self.tool_dropdown.value = options[0][1]
                    self._rebuild_form(self.tool_dropdown.value)
                self.output_area.clear_output(wait=True)
                display(
                    HTML(
                        f"{self._base_styles()}<div class='mcp-ui-wrap' style='color:#047857;"
                        "font-weight:600;font-size:13px;'>"
                        f"✅ Session synchronized (<code>mcp-session-id="
                        f"{html.escape(self.client.session_id or 'stateless')}</code>) — "
                        f"discovered {len(discovered)} tool(s).</div>"
                    )
                )
            except Exception as exc:
                self.output_area.clear_output(wait=True)
                display(
                    HTML(
                        f"{self._base_styles()}<div class='mcp-ui-wrap' style='color:#dc2626;"
                        f"font-weight:600;'>❌ Handshake error: {html.escape(str(exc))}</div>"
                    )
                )

    def _on_execute_clicked(self, _: Any) -> None:
        with self.output_area:
            self.output_area.clear_output(wait=True)
            try:
                tool_name, tool_args = self._collect_arguments()
            except Exception as exc:
                display(
                    HTML(
                        f"{self._base_styles()}<div class='mcp-ui-wrap' style='color:#dc2626;"
                        f"font-weight:600;'>❌ Input validation error: {html.escape(str(exc))}</div>"
                    )
                )
                return

            display(
                HTML(
                    f"{self._base_styles()}<div class='mcp-ui-wrap' style='color:{self.accent_color};"
                    f"font-weight:600;font-size:13px;'>⏳ Invoking <code>tools/call</code> for "
                    f"<b>{html.escape(tool_name)}</b>...</div>"
                )
            )
            try:
                outcome = self.client.call_tool(tool_name, tool_args)
                self.banner_widget.value = self._render_banner_html()
                self.output_area.clear_output(wait=True)
                display(HTML(self._render_call_outcome_html(outcome)))
            except Exception as exc:
                self.output_area.clear_output(wait=True)
                display(
                    HTML(
                        f"{self._base_styles()}<div class='mcp-ui-wrap' style='color:#dc2626;"
                        f"font-weight:600;'>❌ MCP <code>tools/call</code> failed: "
                        f"{html.escape(str(exc))}</div>"
                    )
                )

    def render(self) -> None:
        """Render the complete MCP Web Client interface inside Colab."""
        controls_row = widgets.HBox(
            [self.tool_dropdown, self.refresh_btn, self.auth_btn],
            layout=widgets.Layout(align_items="center", gap="10px", margin="4px 0 10px 0"),
        )
        actions_row = widgets.HBox(
            [self.execute_btn, self.clear_btn],
            layout=widgets.Layout(align_items="center", gap="10px", margin="6px 0 10px 0"),
        )
        display(
            widgets.VBox(
                [
                    self.banner_widget,
                    controls_row,
                    self.tool_header_widget,
                    self.params_container,
                    actions_row,
                    self.output_area,
                ],
                layout=widgets.Layout(width="100%", max_width="980px"),
            )
        )


# ============================================================================
# Initialize & Launch the Loans MCP Web Client
# ============================================================================
if not globals().get("HOSTNAME"):
    raise RuntimeError("❌ HOSTNAME is empty. Please run Step 0 first.")

LOAN_PROXY_NAME = "mcp-gateway-loan-product"
loan_active_rev = (
    wait_for_proxy_ready(LOAN_PROXY_NAME, APIGEE_ENV)
    if "wait_for_proxy_ready" in globals()
    else "latest"
)

loan_mcp_client = MCPStreamableClient(
    mcp_url=f"https://{HOSTNAME}/loanservice/mcp",
    oauth_url=f"https://{HOSTNAME}/v1/oauth/token",
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    auth_mode="oauth2",
)
loan_mcp_client.bootstrap(refresh_auth=True)

loan_workbench = MCPWorkbenchUI(
    client=loan_mcp_client,
    title="🏦 Apigee MCP Gateway — Loans Service Explorer",
    subtitle="Standard MCP Streamable HTTP · OAuth 2.0 Bearer Auth · Dynamic Schema Form · UTF-8 Verified",
    proxy_name=LOAN_PROXY_NAME,
    active_revision=loan_active_rev,
    accent_color="#2563eb",
    accent_dark="#1e3a8a",
)
loan_workbench.render()

## Step 2.1 — Deploy the Payments MCP Gateway, API Product & Developer App

### What This Step Does
Deploys the **Payments MCP Gateway Proxy** (`mcp-gateway-payment-product`), the **Payments API Product** (`ai-payment-package`), and the **Payments Developer Application** (`Payment Agent App`) into your Apigee environment using the Apigee Feature Templater (`aft`).

### Why It Is Necessary
1. **The Payments MCP Proxy** exposes `/paymentservice/mcp` on Apigee, verifies API keys (`x-api-key`) at the gateway edge, validates MCP JSON-RPC 2.0 payloads (`PP-ParsePayload`), and authenticates to the backend Cloud Run Payments MCP server using a Google ID Token.
2. **The Payments API Product** governs which payment operations and environments the agent is permitted to access.
3. **The Payments Developer App** provisions the `API_KEY` (`payment-agent-app-key-123`) and `CLIENT_SECRET` (`payment-agent-app-secret-123`) used by payment agents.

### Inputs & Outputs
- **What You Need to Do (Inputs):** Click **Run** on the code cell below. It reuses the project, environment, and Service Account validated in Step 0.
- **What You Will Get (Outputs):** Confirmation that `mcp-gateway-payment-product` is deployed and in the `READY` state, along with the live URL for the Payments MCP endpoint (`/paymentservice/mcp`).

In [ ]:
#@title 2.1 Deploy MCP Proxy for Payments, API Product & Developer App via AFT
# 1) Deploys templates/mcp-gateway-payment-product.yaml via AFT and waits until READY
# 2) Deploys config/products/ai-payment-agent-package.yaml via AFT
# 3) Deploys config/apps/user_apps_mcp_payments.yaml via AFT

PAYMENT_PROXY_NAME = "mcp-gateway-payment-product"
PAYMENT_TEMPLATE_PATH = "templates/mcp-gateway-payment-product.yaml"
PAYMENT_ENDPOINT_FEATURE_PATH = "features/mcp-payment-endpoint.yaml"
PAYMENT_PRODUCT_PATH = "config/products/ai-payment-agent-package.yaml"
PAYMENT_APP_PATH = "config/apps/user_apps_mcp_payments.yaml"

PAYMENT_ENDPOINT_FEATURE_YAML = """name: mcp-proxy-payment
displayName: mcp-proxy-payment
type: feature
description: mcp-proxy-paymentservice
documentation: ""
gateway: apigee
schemaVersion: 1.0.0
categories: []
parameters: []
endpoints:
  - name: default
    basePath: /paymentservice/mcp
    routes:
      - name: default
        target: default
    flows:
      - name: PreFlow
        mode: Request
        steps:
          - name: PP-ParsePayload
    faultRules: []
targets:
  - name: default
    url: https://mcp-payments-bbva-1052672297239.europe-west1.run.app/mcp
    flows: []
    faultRules: []
    httpTargetConnection:
      Properties:
        Property:
          - metadata:
              name: response.streaming.enabled
            _text: "true"
          - metadata:
              name: request.streaming.enabled
            _text: "true"
      Authentication:
        GoogleIDToken:
          Audience: https://mcp-payments-bbva-1052672297239.europe-west1.run.app
      URL: https://mcp-payments-bbva-1052672297239.europe-west1.run.app/mcp
policies:
  - name: PP-ParsePayload
    type: ParsePayload
    content:
      ParsePayload:
        metadata:
          async: "false"
          continueOnError: "false"
          enabled: "true"
          name: PP-ParsePayload
        DisplayName: PP-ParsePayload
        Source: request
        PayloadType: JSON-RPC-2.0
        Protocol: MCP
resources: []
tests: []
"""

# Ensure payment endpoint feature file is valid before running AFT
payment_feature_file = REPO_DIR / PAYMENT_ENDPOINT_FEATURE_PATH
if (
    not payment_feature_file.is_file()
    or "type: template" in payment_feature_file.read_text(encoding="utf-8")
):
    payment_feature_file.parent.mkdir(parents=True, exist_ok=True)
    payment_feature_file.write_text(PAYMENT_ENDPOINT_FEATURE_YAML, encoding="utf-8")

ensure_repo_files([
    PAYMENT_TEMPLATE_PATH,
    PAYMENT_ENDPOINT_FEATURE_PATH,
    "features/auth-apikey-verify.yaml",
    PAYMENT_PRODUCT_PATH,
    PAYMENT_APP_PATH,
])

# 1. Deploy the MCP Payments Proxy
run_aft_deploy(PAYMENT_TEMPLATE_PATH, "[1/3] Deploying MCP Payments Proxy")
payment_revision = wait_for_proxy_ready(PAYMENT_PROXY_NAME, APIGEE_ENV)

# 2. Deploy the Payments API Product / Package
ensure_env_in_product_yaml(PAYMENT_PRODUCT_PATH, APIGEE_ENV)
run_aft_deploy(PAYMENT_PRODUCT_PATH, "[2/3] Deploying Payments API Product / Package")
print(f"✅ Deployed API Product: {PAYMENT_PRODUCT_PATH}")

# 3. Deploy the Payments Developer & App
run_aft_deploy(PAYMENT_APP_PATH, "[3/3] Deploying Payments Developer & App")
print(f"✅ Deployed Developer & App: {PAYMENT_APP_PATH}")

MCP_PAYMENT_ENDPOINT = f"https://{HOSTNAME}/paymentservice/mcp"
os.environ.update({
    "PROXY_NAME_MCP_PAYMENT": PAYMENT_PROXY_NAME,
    "MCP_PAYMENT_ENDPOINT": MCP_PAYMENT_ENDPOINT,
})

print("=" * 72)
print(f"✅ Deployed MCP Proxy    : {PAYMENT_PROXY_NAME} (rev={payment_revision})")
print(f"✅ API Product           : ai-payment-package ({PAYMENT_PRODUCT_PATH})")
print(f"✅ Developer App         : Payment Agent App (payment-agent@example.com)")
print(f"✅ MCP Payments Endpoint : {MCP_PAYMENT_ENDPOINT}")
print("=" * 72)

## Step 2.2 — Test the Payments MCP Gateway with the Interactive Web Client

### What This Step Does
Launches an interactive **Payments MCP Web Client** inside Google Colab. It connects to `/paymentservice/mcp` using the **Payment Agent App** credentials (`x-api-key` authentication with optional OAuth 2.0 switching), runs the standard MCP handshake (`initialize` → `notifications/initialized` → `tools/list`), and dynamically renders input controls for every discovered payment tool.

### Why It Is Necessary
Confirms that the Payments MCP Gateway proxy properly validates API keys, enforces API Product authorization, and accurately transmits UTF-8 characters (such as `€`, `$`, `£`, beneficiary names, and payment descriptions) between the AI agent client and the backend Payments MCP server.

### Inputs & Outputs
- **What You Need to Do (Inputs):**
  1. Keep the default `API_KEY` and `CLIENT_SECRET` (or enter custom credentials) and run the cell.
  2. Select any discovered payment tool from the **1. Tool** dropdown, enter your desired payment parameters in the dynamically generated form, and click **`▶ Execute Tool (tools/call)`**.
- **What You Will Get (Outputs):**
  - A live **Connection & Handshake Banner** showing each MCP protocol stage, HTTP status, and latency.
  - A **Formatted Tool Output** card displaying the payment result in clean UTF-8, along with an expandable **JSON-RPC 2.0 Request & Response** inspector.

In [ ]:
#@title 2.2 Test MCP Payments Proxy (Standard MCP Protocol & Interactive Web Client)
#@markdown ### App Credentials (`Payment Agent App` — supports `x-api-key` & OAuth 2.0)
API_KEY = "payment-agent-app-key-123" #@param {type:"string"}
CLIENT_SECRET = "payment-agent-app-secret-123" #@param {type:"string"}

if not globals().get("HOSTNAME"):
    raise RuntimeError("❌ HOSTNAME is empty. Please run Step 0 first.")
if "MCPStreamableClient" not in globals() or "MCPWorkbenchUI" not in globals():
    raise RuntimeError(
        "❌ Shared MCP client classes not loaded. Please run Step 1.2 once to load "
        "MCPStreamableClient and MCPWorkbenchUI."
    )

PAYMENT_PROXY_NAME = "mcp-gateway-payment-product"
payment_active_rev = (
    wait_for_proxy_ready(PAYMENT_PROXY_NAME, APIGEE_ENV)
    if "wait_for_proxy_ready" in globals()
    else "latest"
)

payment_mcp_client = MCPStreamableClient(
    mcp_url=f"https://{HOSTNAME}/paymentservice/mcp",
    oauth_url=f"https://{HOSTNAME}/v1/oauth/token",
    client_id=API_KEY,
    client_secret=CLIENT_SECRET,
    auth_mode="x-api-key",
)
payment_mcp_client.bootstrap(refresh_auth=False)

payment_workbench = MCPWorkbenchUI(
    client=payment_mcp_client,
    title="💳 Apigee MCP Gateway — Payments Service Explorer",
    subtitle="Standard MCP Streamable HTTP · API Key & OAuth 2.0 Auth · Dynamic Schema Form · UTF-8 Verified",
    proxy_name=PAYMENT_PROXY_NAME,
    active_revision=payment_active_rev,
    accent_color="#059669",
    accent_dark="#064e3b",
)
payment_workbench.render()